# 1 - How well does DroneMegaDetector do on NOAA footage?

This is the measurement that decides the rest of the project. Before any
fine-tuning, before gathering a single extra image: run the stock model on
real footage and look at what comes out.

[DroneMegaDetector](https://github.com/ConservationDronesAI/DroneMegaDetector)
is an RF-DETR Medium checkpoint trained on drone surveys at roughly 50-120 m
AGL - the Mini 2's range. Three classes: `person`, `animal`, `vehicle`.

Two things to hold in mind while reading the results:

- **Its `animal` class is effectively `mammal`.** The paper is titled
  "Detecting Large Mammals" and the repo says behaviour on non-mammals is
  undefined. Seals should be comfortably in scope. Birds are the known gap -
  that is what notebook 3 exists for.
- **It has never seen a Californian beach.** Whatever it scores here is a
  statement about transfer, not about its quality.

**Before you run this** you need a labelled NOAA evaluation set:

```bash
python prepare_noaa.py --bundle ~/aerial-data/noaa/bundle --out ~/aerial-data/noaa_eval
# label the frames locally in Label Studio or CVAT, export COCO over
# _annotations.coco.json, then come back here
```

> Environment trouble? Run `import doctor; doctor.check(fix=True)` in a
> cell, or `python doctor.py --fix` from the shell. It checks Python,
> FFmpeg, the packages, the GPU and the weights in one pass.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import dmd
import matplotlib.pyplot as plt
from dmd import DMD_CLASSES, load_coco, load_model, predict_images

DATA = Path.home() / "aerial-data"
print("classes:", DMD_CLASSES)

## What is actually in the checkpoint

Not what the README says it is - what the weights say they are. `describe_checkpoint`
reads the config out of the `.pth` without unpickling it.

The class **order** is the part worth checking. It is the interface between
this research directory and `backend/lifejacket/drones/detector.py`, and
getting it wrong relabels every detection silently rather than failing.

In [ ]:
weights = dmd.ensure_dmd_weights()          # downloads once, verifies the SHA-256
info = dmd.describe_checkpoint(weights)
for key, value in info.items():
    print(f"  {key:14s} {value}")

assert tuple(info["class_names"]) == DMD_CLASSES, "class order drift - stop and investigate"
print("\nclass order matches dmd.DMD_CLASSES")

## Load the evaluation set

`load_coco` matches categories by **name**, not by id, because a labelling
tool numbers them however it likes.

In [ ]:
EVAL = DATA / "noaa_eval" / "_annotations.coco.json"

truth = load_coco(EVAL)
print(f"{len(truth)} images, {truth.box_count} boxes")
print("ground truth per class:", truth.per_class_counts())

image_ids = sorted(truth.images)
paths = [truth.image_path(i) for i in image_ids]
missing = [p for p in paths if not p.exists()]
print(f"missing image files: {len(missing)}")

## Are the animals even big enough to detect?

Run this *before* you look at any score. A detector needs roughly **32 px**
on the longest side to work with. If your ground-truth boxes sit below that,
no amount of fine-tuning will help and the answer is to fly lower or raise
the input resolution.

For reference, on the Mini 2 at 50 m the ground width is about 88 m, so a 2 m
sea lion is roughly 44 px across in a 1920 px frame - and about 15 px if the
frame has been downscaled to 640.

In [ ]:
sizes = dmd.pixels_on_target([b for boxes in truth.boxes.values() for b in boxes])

figure, axis = plt.subplots(figsize=(7, 4))
axis.hist(sizes, bins=30, color="#81C784", edgecolor="#555")
axis.axvline(32, color="#C62828", linestyle="--", label="~32 px detector floor")
axis.set_xlabel("longest side of a ground-truth box (px)")
axis.set_ylabel("boxes")
axis.legend()
axis.set_title("pixels on target")
plt.show()

under = sum(1 for s in sizes if s < 32)
print(f"{under} of {len(sizes)} boxes ({under / max(1, len(sizes)):.0%}) are under 32 px")
if under / max(1, len(sizes)) > 0.3:
    print("\nMost targets are below the floor. Expect poor recall, and read it as a")
    print("resolution problem rather than a model problem.")

## Run the model

Predict once at a permissive threshold and filter afterwards, rather than
re-running inference for every threshold you want to try.

In [ ]:
model = load_model()              # stock DroneMegaDetector
predictions = predict_images(model, paths, threshold=0.05)

total = sum(len(v) for v in predictions.values())
print(f"{total} raw detections across {len(paths)} images")

## Look at it before you score it

Twenty annotated frames tell you more than any single number. White outlines
are ground truth; coloured boxes are predictions with their scores.

What to look for:
- Boxes in roughly the right places but consistently offset -> a preprocessing bug
- Animals found but labelled `person` -> a class-order problem
- Nothing at all on obvious animals -> domain gap, or targets too small
- Boxes on waves, foam and sand -> the classic coastal false positive

In [ ]:
PREVIEW_THRESHOLD = 0.3
sample = image_ids[:6]

images = [
    dmd.draw_boxes(
        truth.image_path(i),
        predictions[truth.image_path(i)],
        truth.boxes.get(i, []),
        threshold=PREVIEW_THRESHOLD,
    )
    for i in sample
]
titles = [
    f"{truth.image_path(i).name}  gt={len(truth.boxes.get(i, []))}  "
    f"pred={sum(1 for b in predictions[truth.image_path(i)] if b.score >= PREVIEW_THRESHOLD)}"
    for i in sample
]
dmd.show_grid(images, columns=3, titles=titles)
plt.show()

## Score it

`blind_images` is the column to read first: frames that contain the target
where **nothing at all** was detected. That is the failure a coordinator
would actually feel, and it is more honest than mAP about whether the model
is usable.

In [ ]:
per_image = [(predictions[truth.image_path(i)], truth.boxes.get(i, [])) for i in image_ids]
overall = dmd.sweep(per_image, iou_threshold=0.5)

print(dmd.format_sweep(overall))
best = dmd.best_row(overall)
print(f"\nbest F1 {best['f1']:.3f} at confidence {best['confidence']:.2f}")

dmd.plot_sweep(overall, "stock DroneMegaDetector on NOAA frames")
plt.show()

## Per class

`animal` is the one that matters for a stranding. `person` and `vehicle`
matter for the scene summary. A class with very few ground-truth boxes gives
a number you should not trust - the count is printed so you can see that.

In [ ]:
rows = []
for class_id, name in enumerate(DMD_CLASSES):
    count = sum(1 for boxes in truth.boxes.values() for b in boxes if b.class_id == class_id)
    if count == 0:
        print(f"{name}: no ground truth, skipping")
        continue
    sweep_rows = dmd.sweep(
        [
            (
                [b for b in predictions[truth.image_path(i)] if b.class_id == class_id],
                [b for b in truth.boxes.get(i, []) if b.class_id == class_id],
            )
            for i in image_ids
        ],
        iou_threshold=0.5,
    )
    best_class = dmd.best_row(sweep_rows)
    rows.append((name, count, best_class))
    print(f"\n--- {name} ({count} boxes) ---")
    print(dmd.format_sweep(sweep_rows))

print("\n  class        n      conf    P       R       F1      blind")
for name, count, b in rows:
    print(f"  {name:12s} {count:<6d} {b['confidence']:<7.2f} {b['precision']:<7.3f} "
          f"{b['recall']:<7.3f} {b['f1']:<7.3f} {b['blind_images']}")

## Where is it failing?

Frames with ground truth and no detection, worst first. These are the ones to
open and look at - they tell you whether the gap is size, appearance,
background, or something mundane like a mislabelled frame.

In [ ]:
THRESHOLD = best["confidence"]

misses = []
for i in image_ids:
    gt = truth.boxes.get(i, [])
    found = [b for b in predictions[truth.image_path(i)] if b.score >= THRESHOLD]
    if gt and not found:
        misses.append(i)

print(f"{len(misses)} of {len(image_ids)} frames have ground truth "
      f"but no detection at {THRESHOLD:.2f}")

if misses:
    shown = misses[:6]
    dmd.show_grid(
        [dmd.draw_boxes(truth.image_path(i), predictions[truth.image_path(i)],
                        truth.boxes.get(i, []), threshold=0.05) for i in shown],
        columns=3,
        titles=[f"{truth.image_path(i).name} (all preds >= 0.05)" for i in shown],
    )
    plt.show()

## Save the result

This is the baseline every fine-tune has to beat. Keep it.

In [ ]:
dmd.save_json(
    {
        "tag": "noaa-dmd",
        "weights": "dmd",
        "data": str(EVAL),
        "images": len(paths),
        "ground_truth": truth.per_class_counts(),
        "best": best,
        "sweep": overall,
        "per_class": {name: b for name, _, b in rows},
    },
    dmd.RESULTS / "eval_noaa-dmd.json",
)

## What to conclude

**If recall is decent (say above 0.6) on `animal`:** the pretraining transfers.
Skip straight to fine-tuning on your own footage - notebook 2. You have just
saved yourself the entire pooled-pretraining phase.

**If recall is poor but the boxes are in sensible places:** a domain gap that
fine-tuning closes. That is the normal case and the fine-tune notebooks are
exactly the fix.

**If the model finds nothing and the pixels-on-target histogram was low:**
this is a resolution problem, not a model problem. Fly lower, or raise the
input resolution, before touching the weights.

**If the boxes are systematically offset:** stop and find the preprocessing
bug. Fine-tuning on top of a coordinate error produces a model that is wrong
in a harder-to-see way.

A reminder on what this number is and is not: greedy IoU matching at 0.5, not
COCO mAP, on a small hand-labelled set. It is a go/no-go measurement. It is
not a benchmark and should not be reported as one.